#**pretraining phase**

-> **LLM** is trained on a huge amount of text data.

-> Data is collected, cleaned, and tokenized.

-> The model learns language patterns, grammar, facts, and context.

-> It predicts the next token/word from the given context.

-> Training is repeated over billions/trillions of tokens.

-> Model parameters are updated using backpropagation + gradient descent.

-> Requires large computational resources (GPUs/TPUs).

-> Output is a pre-trained LLM, which can later be fine-tuned or instruction-tuned.


#**supervised fine-tuning phase**

-> Uses a pre-trained LLM.

-> Trains it on labeled instruction–response data.

-> Model learns to follow user instructions.

-> Human-written input → ideal output pairs are used.

-> Updates model weights to improve desired behavior.

-> Makes the LLM more helpful, accurate, and task-specific.

-> Usually comes after pre-training.

#**reforcement learning from human feedback phase**

-> Uses human feedback to improve the LLM.

-> Humans rank or rate model responses.

-> A reward model learns these human preferences.

-> The LLM is trained to produce higher-reward responses.

-> Improves helpfulness, safety, and alignment.

-> Usually comes after Supervised Fine-Tuning (SFT).

# Lab 1 - What is an Agent?  (built by hand, no framework)

An **agent** is not magic. It is:

> **an LLM + some functions you give it + a loop** that runs until the LLM has an answer.

The LLM cannot run code or browse the web itself. What it *can* do is look at a list of
function descriptions you send it and reply *"please call `web_search` with query='...'"*.
**Your code** actually runs the function and hands the result back. Repeat until done.

In this lab we wire that up **manually** with the raw Groq API - hand-written JSON function
schema, hand-written loop - so you see every moving part. Lab 2 does the same thing with
LangChain in one line.

```
  you  -->  [ LLM  +  tool schemas ]
              |
              |  "call web_search(query='...')"
              v
         your code runs web_search()   -->  result
              |
              v
        [ LLM sees the result ]  -->  final answer   (or another tool call -> loop)
```


## Step 0 - Install

In [1]:
%pip install -q groq ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 98.9 MB/s eta 0:00:00


## Step 1 - Groq API key

Free key: https://console.groq.com/keys . Colab: add a secret named `GROQ_API_KEY` (key icon, left sidebar).

In [2]:
import os

def load_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"{name}: from Colab secret"); return v
    except Exception:
        pass
    if os.getenv(name):
        print(f"{name}: from environment"); return os.environ[name]
    from getpass import getpass
    return getpass(f"Paste {name}: ")

GROQ_API_KEY = load_key("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

from groq import Groq
client = Groq(api_key=GROQ_API_KEY)
# A model that is reliable at function calling. (gpt-oss models on Groq tend to emit their
# own built-in "browser" tool calls, which breaks a hand-rolled loop - Qwen does not.)
MODEL = "qwen/qwen3.8-27b"
print(client.chat.completions.create(model=MODEL,
      messages=[{"role":"user","content":"Reply with one word: ready"}]).choices[0].message.content)

GROQ_API_KEY: from Colab secret
Ready


## Step 2 - The limitation: a plain LLM can't look things up

Ask about something recent / live. The model was trained months ago - it will hedge, guess,
or say it doesn't know.

In [3]:
q = "What is the latest stable version of Python, and when was it released?"
plain = client.chat.completions.create(model=MODEL, messages=[{"role":"user","content":q}])
print(plain.choices[0].message.content)

As of late 2024, the **latest stable version** of Python (for general use) is:

### **Python 3.12.x**
- **Latest patch release**: **Python 3.12.8** (released in **October 2024** or shortly before, depending on current date).
- **First release of 3.12**: **October 2, 2023**

However, note that:
- **Python 3.13** is currently in **beta** (as of late 2024), with its official stable release scheduled for **October 2024**.
- If **Python 3.13** has already been released (i.e., if today’s date is on or after **October 7, 2024**), then **Python 3.13.0** is the latest stable version.

### For the most accurate and up-to-date information:
Check the official Python releases page:
🔗 [https://www.python.org/downloads/](https://www.python.org/downloads/)

### Summary (as of late October 2024):
| Version       | Status     | Release Date        |
|---------------|------------|---------------------|
| Python 3.13   | Stable     | **October 7, 2024** |
| Python 3.12   | Stable     | October 2, 2023    

## Step 3 - Write a real tool (just a Python function)

`web_search` takes a query string and returns text. Nothing LLM-specific here yet - it is a
normal function you could call yourself.

In [5]:
import time, re, requests
from ddgs import DDGS

def web_search(query: str, max_results: int = 5) -> str:
    """Run a web search and return the top results as text.
    Tries DuckDuckGo; if it is rate-limited, falls back to Wikipedia search."""
    for _ in range(3):
        try:
            hits = list(DDGS().text(query, max_results=max_results))
            if hits:
                return "\n\n".join(f"- {h['title']}\n  {h['body']}\n  ({h['href']})" for h in hits)
        except Exception:
            pass
        time.sleep(2)
    try:  # fallback
        r = requests.get("https://en.wikipedia.org/w/api.php", timeout=15,
            headers={"User-Agent": "GSSS-Agent-Lab/1.0 (teaching notebook)"}, params={
            "action": "query", "list": "search", "srsearch": query,
            "format": "json", "srlimit": max_results})
        items = r.json()["query"]["search"]
        return "\n\n".join(f"- {x['title']}\n  {re.sub('<[^>]+>', '', x['snippet'])}" for x in items) or "No results."
    except Exception as e:
        return f"search unavailable: {e}"

print(web_search("latest stable Python version release date"))

- History of Python - Wikipedia
  3 weeks ago - Many of its major features were ... Python versions 2.6 and 2.7 until support for Python 2 finally ceased at the beginning of 2020. Releases of Python 3 up through 3.12 include the 2to3 utility, which automates the translation of Python 2 code to Python 3. As of June 2026, Python 3.14.6 is the latest stable ...
  (https://en.wikipedia.org/wiki/History_of_Python)

- Status of Python versions
  May 27, 2026 - The main branch is currently the future Python 3.16, and is the only branch that accepts new features. The latest release for each Python version can be found on the download page.
  (https://devguide.python.org/versions/)

- Python documentation by version | Python.org
  Python 1.4, released on 25 October 1996 · The latest, and unreleased, documentation for versions of Python still under development: Development version · Python 3.x · The Python Software Foundation is the organization behind Python.
  (https://www.python.org/doc/versi

## Step 4 - Describe the tool to the LLM (the hand-written JSON schema)

This dict is the **only** thing the model knows about your function. The `description`
fields are read by the model to decide *when* and *how* to call it - write them like you
are explaining to a new teammate.

In [6]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the public web for current information. Use for anything "
                           "recent, factual, or that you are not sure about.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "the search query"}
                },
                "required": ["query"],
            },
        },
    }
]

# a name -> function map so our code can actually run what the model asks for
TOOL_FUNCTIONS = {"web_search": web_search}
print("Declared tools:", [t["function"]["name"] for t in TOOLS_SCHEMA])

Declared tools: ['web_search']


In [7]:
q

'What is the latest stable version of Python, and when was it released?'

## Step 5 - One round-trip, done by hand

Watch the three messages: (1) our question, (2) the model's *tool call* (no answer yet),
(3) we run the function and send back a `role="tool"` message, then the model answers.

In [8]:
import json

messages = [{"role": "user", "content": "What is the capital of India"}]

# --- call 1: the model decides to use a tool ---
resp = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS_SCHEMA)
choice = resp.choices[0].message
print( resp.choices[0].finish_reason)
print("MODEL wants tool calls:", bool(choice.tool_calls))
for tc in choice.tool_calls or []:
    print(f"  -> {tc.function.name}({tc.function.arguments})")

# # append the model's tool-call message
# messages.append(choice)

# # --- run each requested tool, append its result ---
# for tc in choice.tool_calls:
#     args = json.loads(tc.function.arguments)
#     result = TOOL_FUNCTIONS[tc.function.name](**args)
#     messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
#     print(f"\n  ran {tc.function.name} -> {result[:150]}...")

# # --- call 2: no tools this time, so the model just writes the answer ---
# final = client.chat.completions.create(model=MODEL, messages=messages)
# print("\nFINAL ANSWER:\n", final.choices[0].message.content)

stop
MODEL wants tool calls: False


## Step 6 - The agent loop

A single round-trip only handles one tool call. Real questions need several ("search this,
then calculate that"). So we put it in a **loop**, capped at `MAX_STEPS` so it always
stops. We also add a second tool - `calculator` - so the model has a real choice to make.

In [9]:
def calculator(expression: str) -> str:
    """Evaluate a arithmetic expression like '(1234 * 2) / 7'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {e}"

TOOLS_SCHEMA = [
    TOOLS_SCHEMA[0],
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate an arithmetic expression. Use this instead of doing "
                           "mental math.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    },
]
TOOL_FUNCTIONS = {"web_search": web_search, "calculator": calculator}


def _final_answer(messages):
    """Ask the model to answer with no tools available - always returns text."""
    return client.chat.completions.create(model=MODEL, messages=messages).choices[0].message.content

def run_agent(question: str, max_steps: int = 5, verbose: bool = True):
    messages = [
        {"role": "system", "content": "You are a helpful agent. Use tools when useful. "
                                      "Think step by step and call one tool at a time."},
        {"role": "user", "content": question},
    ]
    trace = []
    for step in range(1, max_steps + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL, messages=messages, tools=TOOLS_SCHEMA, tool_choice="auto")
        except Exception as e:
            # gpt-oss occasionally emits a malformed tool call -> Groq 400s.
            # Recover by asking for a plain answer with what we have so far.
            if verbose:
                print(f"[step {step}] bad tool call ({str(e)[:60]}...) -> forcing a plain answer")
            return {"answer": _final_answer(messages), "trace": trace}

        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            if verbose:
                print(f"[step {step}] final answer")
            return {"answer": msg.content, "trace": trace}

        for tc in msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
                result = TOOL_FUNCTIONS[tc.function.name](**args)
            except Exception as e:
                args, result = {}, f"tool error: {e}"
            trace.append((tc.function.name, args, result))
            if verbose:
                print(f"[step {step}] {tc.function.name}({args}) -> {str(result)[:120]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

    # ran out of steps - make one last no-tools call so the user still gets an answer
    return {"answer": _final_answer(messages), "trace": trace}


out = run_agent("What is the population of the capital city of Japan, divided by 12?")
print("\nANSWER:", out["answer"])

[step 1] web_search({'query': 'population of Tokyo Japan capital city 2024'}) -> - Tokyo - Wikipedia
  Tokyo, [a] officially the Tokyo Metropolis,[b] is the capital and most populous city of Japan. The
[step 2] calculator({'expression': '14192184 / 12'}) -> 1182682.0
[step 3] final answer

ANSWER: The population of Tokyo, Japan's capital city, is approximately **14,192,184** (as of 2024). Divided by 12, that equals **1,182,682**.


## Step 7 - Another multi-step question

In [10]:
_ = run_agent("Who won the most recent Formula 1 world championship, and what year did that driver make their F1 debut? Give the gap in years.")

[step 1] web_search({'query': 'most recent Formula 1 world champion 2024 2025'}) -> - 2025 Formula One World Championship - Wikipedia
  2 weeks ago - Jack Doohan, who replaced Ocon for the 2024 Abu Dhabi 
[step 2] web_search({'query': 'Lando Norris F1 debut year Ferrari 2019'}) -> - Lando Norris - Wikipedia
  1 day ago - ↑ "Leclerc thrills the Tifosi ... bold Ferrari strategy paying off". Formula1.c
[step 3] calculator({'expression': '2025 - 2019'}) -> 6
[step 4] final answer


## Step 8 - Gradio app

Chat with the hand-built agent. The panel shows every tool call it made.

In [11]:
import gradio as gr

def agent_ui(question, history):
    out = run_agent(question, verbose=False)
    if out["trace"]:
        tr = "\n".join(f"{i+1}. {name}({args}) -> {str(res)[:200]}"
                        for i, (name, args, res) in enumerate(out["trace"]))
    else:
        tr = "(answered with no tools)"
    return out["answer"], tr

with gr.Blocks(title="Lab 1 - Hand-built Agent") as demo:
    gr.Markdown("# Lab 1 - Agent from scratch (no framework)\nWeb search + calculator, wired by hand.")
    q = gr.Textbox(label="Ask the agent", value="What is the latest stable version of Python?")
    btn = gr.Button("Run", variant="primary")
    ans = gr.Textbox(label="Answer", lines=4)
    trace = gr.Textbox(label="Tool calls the agent made", lines=10)
    gr.Examples([
        "What is the latest stable version of Python?",
        "What is the population of the capital of France, divided by 1000?",
        "Who is the current CEO of Nintendo?",
    ], inputs=q)
    btn.click(agent_ui, [q, ans], [ans, trace])

demo.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c41ef37cb61552bc52.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Recap

- An agent = **LLM + tools + loop**. We built all three by hand.
- The **JSON schema** is the contract: the model only knows what your `description` says.
- The **loop** is what lets it chain tools; the **cap** is what makes it safe.
- Lab 2 replaces ~40 lines of this with `create_agent(llm, tools)` - same idea, less typing,
  plus a bigger toolbox.

### Exercises
1. Add a `get_current_datetime()` tool (no arguments). Ask "how many days until New Year?"
2. Break `web_search` on purpose (return `"error"`). Does the agent recover or loop?
3. Lower `max_steps` to 1 on the F1 question - where does it get stuck?
